In [1]:
!pip -q install ultralytics datasets huggingface_hub pyarrow pandas opencv-python
!pip -q install yt-dlp
!apt-get -qq update
!apt-get -qq install -y ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.1/182.1 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 119.1 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
!mkdir -p "/content/drive/MyDrive/ai_spring_2026"

YOUTUBE_URL = "https://www.youtube.com/watch?v=YcvECxtXoxQ"
CLIP_PATH = "/content/drive/MyDrive/ai_spring_2026/input_video_18m43s_24m43s.mp4"

# Downloading only 18:43-24:43
!yt-dlp \
  --download-sections "*00:18:43-00:24:43" \
  -f "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]" \
  --merge-output-format mp4 \
  -o "$CLIP_PATH" \
  "$YOUTUBE_URL"

!ls -lh "$CLIP_PATH"

[youtube] Extracting URL: https://www.youtube.com/watch?v=YcvECxtXoxQ
[youtube] YcvECxtXoxQ: Downloading webpage
[youtube] YcvECxtXoxQ: Downloading android vr player API JSON
[info] YcvECxtXoxQ: Downloading 1 format(s): 401+140
[info] YcvECxtXoxQ: Downloading 1 time ranges: 1123.0-1483.0
[download] Destination: /content/drive/MyDrive/ai_spring_2026/input_video_18m43s_24m43s.mp4
[libdav1d @ 0x57715767c180] libdav1d 0.9.2
Input #0, mov,mp4,m4a,3gp,3g2,mj2, from 'https://rr4---sn-npoldn7s.googlevideo.com/videoplayback?expire=1771822840&ei=l4qbabefPJyk9fwPioTvwQg&ip=34.126.106.216&id=o-AGfuQngukik0Hi3RSU3o9p45y3JcvHTSctMA0Ncv2Nv0&itag=401&source=youtube&requiressl=yes&xpc=EgVo2aDSNQ%3D%3D&cps=339&met=1771801239%2C&mh=ca&mm=31%2C29&mn=sn-npoldn7s%2Csn-npoe7ney&ms=au%2Crdu&mv=u&mvi=4&pl=20&rms=au%2Cau&bui=AVNa5-wiWhoGlfPNphgeqrvN89LVB-3yRkIAcNFqAAF-IZs1MWT2J0keYXsLBip3CCx-23gakk_88j2q&spc=6dlaFCSkzwlU&vprv=1&svpuc=1&mime=video%2Fmp4&rqh=1&gir=yes&clen=4575382188&dur=2793.790&lmt=176594925253

In [3]:
FRAMES_DIR = "/content/drive/MyDrive/ai_spring_2026/frames_5s_18m43_24m43"
!mkdir -p "$FRAMES_DIR"

!ffmpeg -hide_banner -loglevel error -i "$CLIP_PATH" -vf "fps=1/5" "$FRAMES_DIR/frame_%06d.jpg"
!ls "$FRAMES_DIR" | head
!ls "$FRAMES_DIR" | wc -l

frame_000001.jpg
frame_000002.jpg
frame_000003.jpg
frame_000004.jpg
frame_000005.jpg
frame_000006.jpg
frame_000007.jpg
frame_000008.jpg
frame_000009.jpg
frame_000010.jpg
72


In [4]:
import os, json, math, re
import pandas as pd
from ultralytics import YOLO
from datasets import load_dataset


VIDEO_ID = "YcvECxtXoxQ"
SECONDS_PER_FRAME = 5

# This clip corresponds to 18:43–24:43 in the ORIGINAL YouTube video
CLIP_START_SEC = 18*60 + 43
CLIP_END_SEC   = 24*60 + 43


CLIP_PATH = "/content/drive/MyDrive/ai_spring_2026/input_video_18m43s_24m43s.mp4"
FRAMES_DIR = "/content/drive/MyDrive/ai_spring_2026/frames_5s_18m43_24m43"

OUT_DIR = "/content/drive/MyDrive/ai_spring_2026/assignment2_interval_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

DETECTIONS_PARQUET = os.path.join(OUT_DIR, "detections_by_frame.parquet")
CLIPS_PARQUET = os.path.join(OUT_DIR, "clips_by_query.parquet")

print("Clip path exists:", os.path.exists(CLIP_PATH), CLIP_PATH)
print("Frames dir exists:", os.path.exists(FRAMES_DIR), FRAMES_DIR)
print("Output dir:", OUT_DIR)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Clip path exists: True /content/drive/MyDrive/ai_spring_2026/input_video_18m43s_24m43s.mp4
Frames dir exists: True /content/drive/MyDrive/ai_spring_2026/frames_5s_18m43_24m43
Output dir: /content/drive/MyDrive/ai_spring_2026/assignment2_interval_outputs


In [5]:
MODEL_WEIGHTS_PATH = os.path.join(OUT_DIR, "carparts_best.pt")
IMGSZ = 640

if os.path.exists(MODEL_WEIGHTS_PATH):
    model = YOLO(MODEL_WEIGHTS_PATH)
    print("Loaded trained weights from Drive:", MODEL_WEIGHTS_PATH)
else:
    base = YOLO("yolo11n-seg.pt")
    results = base.train(data="carparts-seg.yaml", epochs=5, imgsz=IMGSZ)
    best_path = str(results.save_dir / "weights" / "best.pt")
    model = YOLO(best_path)

    import shutil
    shutil.copy(best_path, MODEL_WEIGHTS_PATH)
    print("Trained model saved at:", best_path)
    print("Copied to Drive:", MODEL_WEIGHTS_PATH)

print("Number of classes:", len(model.names))
print("Names:", list(model.names.items()))


Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=carparts-seg.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspec

In [7]:
EXTERIOR_LABELS = {
    "back_bumper",
    "front_bumper",
    "hood",
    "wheel",
    "left_mirror",
    "right_mirror",
    "front_light",
    "back_light",
    "front_left_light",
    "front_right_light",
    "back_left_light",
    "back_right_light",
    "front_glass",
    "back_glass",
    "front_door",
    "back_door",
    "front_left_door",
    "front_right_door",
    "back_left_door",
    "back_right_door",
    "tailgate",
    "trunk",
}

model_label_set = set(model.names.values())
missing = EXTERIOR_LABELS - model_label_set
extra = (model_label_set - EXTERIOR_LABELS) - {"object"}  # label not used for exterior filter


In [8]:
import numpy as np

CONF_TH = 0.25

frame_files = sorted([f for f in os.listdir(FRAMES_DIR) if f.lower().endswith(".jpg")])
print("Total frames found:", len(frame_files))

rows = []
total_dets = 0

for frame_id, fname in enumerate(frame_files):
    img_path = os.path.join(FRAMES_DIR, fname)

    ts_clip = frame_id * SECONDS_PER_FRAME
    ts_original = CLIP_START_SEC + ts_clip

    pred = model.predict(img_path, conf=CONF_TH, verbose=False)[0]
    det_list = []

    if pred.boxes is not None and len(pred.boxes) > 0:
        for b in pred.boxes:
            cls_id = int(b.cls[0].item())
            label = model.names[cls_id]
            if label not in EXTERIOR_LABELS:
                continue

            conf = float(b.conf[0].item())
            x1, y1, x2, y2 = [float(v) for v in b.xyxy[0].tolist()]

            det_list.append({
                "class_label": label,
                "bounding_box": [x1, y1, x2, y2],
                "confidence_score": conf,
                "timestamp_sec": int(ts_original),
            })

    total_dets += len(det_list)
    rows.append({"frame_id": int(frame_id), "detections": det_list})

df_frames = pd.DataFrame(rows)
print("Frames rows:", len(df_frames), "Total exterior detections kept:", total_dets)

df_frames.to_parquet(DETECTIONS_PARQUET, index=False)
print("Saved:", DETECTIONS_PARQUET)
df_frames.head(3)


Total frames found: 72
Frames rows: 72 Total exterior detections kept: 421
Saved: /content/drive/MyDrive/ai_spring_2026/assignment2_interval_outputs/detections_by_frame.parquet


,frame_id,detections
0,0,"[{'class_label': 'front_bumper', 'bounding_box..."
1,1,"[{'class_label': 'left_mirror', 'bounding_box'..."
2,2,"[{'class_label': 'front_glass', 'bounding_box'..."


In [9]:
ds = load_dataset("aegean-ai/rav4-exterior-images", split="train")
print("Query images:", len(ds))

def query_labels(img, conf_th=0.25):
    pred = model.predict(img, conf=conf_th, verbose=False)[0]
    labels = []
    if pred.boxes is not None and len(pred.boxes) > 0:
        for b in pred.boxes:
            label = model.names[int(b.cls[0].item())]
            if label in EXTERIOR_LABELS:
                labels.append(label)
    seen=set()
    out=[]
    for l in labels:
        if l not in seen:
            seen.add(l)
            out.append(l)
    return out

def contiguous_segments(times, step=5, max_gap=5):
    if not times:
        return []
    times = sorted(times)
    segs=[]
    start=prev=times[0]
    for t in times[1:]:
        if t - prev <= max_gap:
            prev = t
        else:
            segs.append((start, prev + step))
            start = prev = t
    segs.append((start, prev + step))
    return segs

label_to_times = {}
label_to_support = {}

for _, row in df_frames.iterrows():
    dets = row["detections"]
    for d in dets:
        lbl = d["class_label"]
        ts = int(d["timestamp_sec"])
        label_to_times.setdefault(lbl, []).append(ts)
        label_to_support[lbl] = label_to_support.get(lbl, 0) + 1

for lbl in list(label_to_times.keys()):
    label_to_times[lbl] = sorted(set(label_to_times[lbl]))

print("Indexed labels:", sorted(label_to_times.keys()))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/505 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/69.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/65 [00:00<?, ? examples/s]

Query images: 65
Indexed labels: ['back_bumper', 'back_door', 'back_glass', 'back_left_door', 'back_left_light', 'back_light', 'back_right_door', 'back_right_light', 'front_bumper', 'front_glass', 'front_left_door', 'front_left_light', 'front_right_door', 'hood', 'left_mirror', 'right_mirror', 'trunk', 'wheel']


In [10]:
clip_rows = []
verify_rows = []

for qi in range(len(ds)):
    img = ds[qi]["image"]
    labels = query_labels(img, conf_th=0.25)
    if not labels:
        continue

    for lbl in labels:
        times = label_to_times.get(lbl, [])
        segs = contiguous_segments(times, step=SECONDS_PER_FRAME, max_gap=SECONDS_PER_FRAME)
        support = int(label_to_support.get(lbl, 0))

        for (s, e) in segs:
            s2 = max(int(s), CLIP_START_SEC)
            e2 = min(int(e), CLIP_END_SEC)

            if e2 <= s2:
                continue

            url = f"https://www.youtube.com/embed/{VIDEO_ID}?start={s2}&end={e2}"
            query_id = f"query_{qi}:{lbl}"

            clip_rows.append({
                "query_image": query_id,
                "youtube_url": url
            })

            verify_rows.append({
                "query_image": query_id,
                "start_timestamp": s2,
                "end_timestamp": e2,
                "class_label_used_for_retrieval": lbl,
                "number_of_supporting_detections": support
            })

clips_df = pd.DataFrame(clip_rows)
clips_df.to_parquet(CLIPS_PARQUET, index=False)
print("Saved:", CLIPS_PARQUET, "rows:", len(clips_df))

clips_df.head(10)


Saved: /content/drive/MyDrive/ai_spring_2026/assignment2_interval_outputs/clips_by_query.parquet rows: 3025


,query_image,youtube_url
0,query_0:front_glass,https://www.youtube.com/embed/YcvECxtXoxQ?star...
1,query_0:front_glass,https://www.youtube.com/embed/YcvECxtXoxQ?star...
2,query_0:front_glass,https://www.youtube.com/embed/YcvECxtXoxQ?star...
3,query_0:front_glass,https://www.youtube.com/embed/YcvECxtXoxQ?star...
4,query_0:front_glass,https://www.youtube.com/embed/YcvECxtXoxQ?star...
5,query_0:front_glass,https://www.youtube.com/embed/YcvECxtXoxQ?star...
6,query_0:front_glass,https://www.youtube.com/embed/YcvECxtXoxQ?star...
7,query_0:front_glass,https://www.youtube.com/embed/YcvECxtXoxQ?star...
8,query_0:wheel,https://www.youtube.com/embed/YcvECxtXoxQ?star...
9,query_0:wheel,https://www.youtube.com/embed/YcvECxtXoxQ?star...


In [12]:
from huggingface_hub import notebook_login, create_repo, upload_file, whoami
notebook_login()
username = whoami()["name"]
repo_id = f"{username}/rav4-exterior-retrieval-interval2"
create_repo(repo_id, repo_type="dataset", exist_ok=True)
upload_file(path_or_fileobj=DETECTIONS_PARQUET, path_in_repo="detections_by_frame.parquet", repo_id=repo_id, repo_type="dataset")
upload_file(path_or_fileobj=CLIPS_PARQUET, path_in_repo="clips_by_query.parquet", repo_id=repo_id, repo_type="dataset")
print("Uploaded:", f"https://huggingface.co/datasets/{repo_id}")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...tections_by_frame.parquet: 100%|##########| 19.8kB / 19.8kB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ts/clips_by_query.parquet: 100%|##########| 7.50kB / 7.50kB            

Uploaded: https://huggingface.co/datasets/Naman02/rav4-exterior-retrieval-interval2


In [14]:
import pandas as pd, json, os

DETECTIONS_PATH = globals().get("DETECTIONS_PARQUET", "/content/drive/MyDrive/ai_spring_2026/assignment2_interval_outputs/detections_by_frame.parquet")

det = pd.read_parquet(DETECTIONS_PATH)

print("DETECTIONS parquet:", DETECTIONS_PATH)
print("Number of rows:", len(det))
print("Columns:", det.columns.tolist())


pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 0)

display(det.head(len(det)))

DETECTIONS parquet: /content/drive/MyDrive/ai_spring_2026/assignment2_interval_outputs/detections_by_frame.parquet
Number of rows: 72
Columns: ['frame_id', 'detections']


,frame_id,detections
0,0,"[{'bounding_box': [1270.9058837890625, 998.694580078125, 2737.08837890625, 1583.21728515625], 'class_label': 'front_bumper', 'confidence_score': 0.7203534245491028, 'timestamp_sec': 1123}, {'bounding_box': [1582.9947509765625, 575.571044921875, 2493.60107421875, 823.366943359375], 'class_label': 'front_glass', 'confidence_score': 0.7059865593910217, 'timestamp_sec': 1123}, {'bounding_box': [2554.46484375, 716.3197631835938, 2671.46923828125, 816.2191772460938], 'class_label': 'right_mirror', 'confidence_score': 0.6728319525718689, 'timestamp_sec': 1123}, {'bounding_box': [2407.58056640625, 789.0181274414062, 2672.759033203125, 1064.6949462890625], 'class_label': 'right_mirror', 'confidence_score': 0.5637875199317932, 'timestamp_sec': 1123}, {'bounding_box': [2551.889404296875, 715.4511108398438, 2672.40869140625, 818.7738647460938], 'class_label': 'left_mirror', 'confidence_score': 0.544740617275238, 'timestamp_sec': 1123}, {'bounding_box': [2411.54638671875, 783.826904296875, 2654.7578125, 953.7978515625], 'class_label': 'right_mirror', 'confidence_score': 0.4824162721633911, 'timestamp_sec': 1123}, {'bounding_box': [1440.64990234375, 694.700439453125, 1554.859619140625, 795.8564453125], 'class_label': 'left_mirror', 'confidence_score': 0.4638976752758026, 'timestamp_sec': 1123}, {'bounding_box': [2414.775146484375, 787.4959716796875, 2662.50146484375, 960.3819580078125], 'class_label': 'left_mirror', 'confidence_score': 0.39379701018333435, 'timestamp_sec': 1123}, {'bounding_box': [1371.0545654296875, 816.3272094726562, 1598.5667724609375, 932.315185546875], 'class_label': 'left_mirror', 'confidence_score': 0.35635918378829956, 'timestamp_sec': 1123}, {'bounding_box': [1372.418212890625, 809.0679931640625, 1598.5400390625, 932.578125], 'class_label': 'right_mirror', 'confidence_score': 0.32921940088272095, 'timestamp_sec': 1123}, {'bounding_box': [2410.1669921875, 794.8053588867188, 2676.2451171875, 1065.1162109375], 'class_label': 'left_mirror', 'confidence_score': 0.2665582299232483, 'timestamp_sec': 1123}]"
1,1,"[{'bounding_box': [2566.1484375, 402.3403015136719, 2725.436279296875, 534.3275146484375], 'class_label': 'left_mirror', 'confidence_score': 0.6023051142692566, 'timestamp_sec': 1128}, {'bounding_box': [1244.348876953125, 249.29193115234375, 2445.05712890625, 468.7339782714844], 'class_label': 'back_glass', 'confidence_score': 0.5621300339698792, 'timestamp_sec': 1128}, {'bounding_box': [566.009033203125, 693.6474609375, 2781.4072265625, 1492.5296630859375], 'class_label': 'front_bumper', 'confidence_score': 0.3954602777957916, 'timestamp_sec': 1128}, {'bounding_box': [2566.95556640625, 401.281494140625, 2720.73779296875, 533.03759765625], 'class_label': 'right_mirror', 'confidence_score': 0.32179832458496094, 'timestamp_sec': 1128}, {'bounding_box': [2266.6875, 1401.102294921875, 2758.607666015625, 1680.2457275390625], 'class_label': 'wheel', 'confidence_score': 0.3044389486312866, 'timestamp_sec': 1128}, {'bounding_box': [641.625, 1412.7813720703125, 2759.53955078125, 1641.226318359375], 'class_label': 'wheel', 'confidence_score': 0.26100483536720276, 'timestamp_sec': 1128}]"
2,2,"[{'bounding_box': [2581.06787109375, 676.914306640625, 3814.171875, 1067.05615234375], 'class_label': 'front_glass', 'confidence_score': 0.8669127821922302, 'timestamp_sec': 1133}, {'bounding_box': [2006.212646484375, 968.2445068359375, 3837.664306640625, 1505.56494140625], 'class_label': 'hood', 'confidence_score': 0.8382201194763184, 'timestamp_sec': 1133}, {'bounding_box': [1941.7203369140625, 1254.49755859375, 3840.0, 2160.0], 'class_label': 'front_bumper', 'confidence_score': 0.3667588531970978, 'timestamp_sec': 1133}, {'bounding_box': [2609.736083984375, 688.5404052734375, 3833.83447265625, 1056.9058837890625], 'class_label': 'back_glass', 'confidence_score': 0.3613055646419525, 'timestamp_sec': 1133}, {'bounding_box': [109.88749694824219, 606.6083984375, 412.3135986328125, 1403.7547607421875], 'class_la

In [15]:
import pandas as pd, os

CLIPS_PATH = globals().get(
    "CLIPS_PARQUET",
    "/content/drive/MyDrive/ai_spring_2026/assignment2_interval_outputs/clips_by_query.parquet"
)

clips = pd.read_parquet(CLIPS_PATH)

print("CLIPS parquet:", CLIPS_PATH)
print("Number of rows:", len(clips))
print("Columns:", clips.columns.tolist())

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 0)

display(clips.head(len(clips)))

CLIPS parquet: /content/drive/MyDrive/ai_spring_2026/assignment2_interval_outputs/clips_by_query.parquet
Number of rows: 3025
Columns: ['query_image', 'youtube_url']


,query_image,youtube_url
0,query_0:front_glass,https://www.youtube.com/embed/YcvECxtXoxQ?start=1123&end=1128
1,query_0:front_glass,https://www.youtube.com/embed/YcvECxtXoxQ?start=1133&end=1143
2,query_0:front_glass,https://www.youtube.com/embed/YcvECxtXoxQ?start=1148&end=1158
3,query_0:front_glass,https://www.youtube.com/embed/YcvECxtXoxQ?start=1163&end=1233
4,query_0:front_glass,https://www.youtube.com/embed/YcvECxtXoxQ?start=1238&end=1253
...,...,...
3020,query_64:back_light,https://www.youtube.com/embed/YcvECxtXoxQ?start=1258&end=1263
3021,query_64:back_light,https://www.youtube.com/embed/YcvECxtXoxQ?start=1268&end=1273
3022,query_64:back_light,https://www.youtube.com/embed/YcvECxtXoxQ?start=1338&end=1348
3023,query_64:back_light,https://www.youtube.com/embed/YcvECxtXoxQ?start=1428&end=1433


In [20]:
import os, json
import pandas as pd
import numpy as np

DETECTIONS_PATH = globals().get(
    "DETECTIONS_PARQUET",
    "/content/drive/MyDrive/ai_spring_2026/assignment2_interval_outputs/detections_by_frame.parquet"
)

OUT_DIR = globals().get(
    "OUT_DIR",
    "/content/drive/MyDrive/ai_spring_2026/assignment2_interval_outputs"
)
os.makedirs(OUT_DIR, exist_ok=True)
BEST_CLIPS_PARQUET = os.path.join(OUT_DIR, "best_clips_by_query.parquet")

det = globals().get("det", None)
if det is None:
    det = pd.read_parquet(DETECTIONS_PATH)

ds = globals().get("ds", None)
if ds is None:
    from datasets import load_dataset
    ds = load_dataset("aegean-ai/rav4-exterior-images", split="train")

model = globals().get("model", None)
if model is None:
    from ultralytics import YOLO
    MODEL_WEIGHTS_PATH = globals().get("MODEL_WEIGHTS_PATH", os.path.join(OUT_DIR, "carparts_best.pt"))
    if os.path.exists(MODEL_WEIGHTS_PATH):
        model = YOLO(MODEL_WEIGHTS_PATH)
    else:
        model = YOLO("yolo11n-seg.pt")

VIDEO_ID = globals().get("VIDEO_ID", "YcvECxtXoxQ")
SECONDS_PER_FRAME = int(globals().get("SECONDS_PER_FRAME", 5))
CONF_TH = float(globals().get("CONF_TH", 0.25))

EXTERIOR_LABELS = globals().get("EXTERIOR_LABELS", None)
is_exterior = globals().get("is_exterior", None)

def _is_exterior_label(lbl: str) -> bool:
    if callable(is_exterior):
        try:
            return bool(is_exterior(lbl))
        except Exception:
            pass
    if isinstance(EXTERIOR_LABELS, (set, list, tuple)):
        return lbl in set(EXTERIOR_LABELS)
    return True

def flatten_frame_detections(det_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, r in det_df.iterrows():
        frame_id = int(r.get("frame_id", r.get("frame_index", 0)))

        dets = r.get("detections", [])
        # ---- FIX: avoid "array truth value ambiguous" ----
        if dets is None:
            dets = []
        elif not isinstance(dets, list):
            # handles numpy arrays / pandas objects
            dets = list(dets)

        for d in dets:
            if d is None:
                continue
            rows.append({
                "frame_id": frame_id,
                "timestamp_sec": int(d.get("timestamp_sec", frame_id * SECONDS_PER_FRAME)),
                "class_label": d.get("class_label", ""),
                "confidence_score": float(d.get("confidence_score", 0.0)),
                "bounding_box": d.get("bounding_box", None),
            })
    return pd.DataFrame(rows)

def contiguous_segments(times, step=5, max_gap=5):
    if len(times) == 0:
        return []
    times = sorted(times)
    segs = []
    start = prev = times[0]
    for t in times[1:]:
        if t - prev <= max_gap:
            prev = t
        else:
            segs.append((start, prev + step))
            start = prev = t
    segs.append((start, prev + step))
    return segs

det_rowwise = flatten_frame_detections(det)
if len(det_rowwise) == 0:
    raise RuntimeError("No detections found in detections_by_frame.parquet; cannot build best_clips_by_query.parquet.")

def best_segment_for_query(query_img, query_idx: int):
    pred = model.predict(query_img, conf=CONF_TH, verbose=False)[0]
    if pred.boxes is None or len(pred.boxes) == 0:
        return None

    q_labels = []
    for b in pred.boxes:
        q_labels.append(model.names[int(b.cls[0].item())])
    q_labels = list(dict.fromkeys(q_labels))
    q_labels = [lbl for lbl in q_labels if _is_exterior_label(lbl)]

    best = None
    for label in q_labels:
        hits = det_rowwise[(det_rowwise["class_label"] == label) & (det_rowwise["confidence_score"] >= CONF_TH)]
        if len(hits) == 0:
            continue

        times = hits["timestamp_sec"].unique().tolist()
        segs = contiguous_segments(times, step=SECONDS_PER_FRAME, max_gap=SECONDS_PER_FRAME)

        for (s, e) in segs:
            seg_hits = hits[(hits["timestamp_sec"] >= s) & (hits["timestamp_sec"] < e)]
            support = int(len(seg_hits))
            if support <= 0:
                continue
            avg_conf = float(seg_hits["confidence_score"].mean())
            duration = int(e - s)
            score = support + 0.1 * avg_conf + 0.01 * duration

            candidate = {
                "query_image": f"query_{query_idx}:{label}",
                "youtube_url": f"https://www.youtube.com/embed/{VIDEO_ID}?start={int(s)}&end={int(e)}",
                "_score": score,
            }
            if (best is None) or (candidate["_score"] > best["_score"]):
                best = candidate

    return best

best_rows = []
for qi in range(len(ds)):
    best = best_segment_for_query(ds[qi]["image"], qi)
    if best is None:
        best_rows.append({"query_image": f"query_{qi}", "youtube_url": ""})
    else:
        best_rows.append({"query_image": best["query_image"], "youtube_url": best["youtube_url"]})

best_clips = pd.DataFrame(best_rows)
best_clips.to_parquet(BEST_CLIPS_PARQUET, index=False)

print("Saved:", BEST_CLIPS_PARQUET)
print("Rows:", len(best_clips))
best_clips.head(len(best_clips))

Saved: /content/drive/MyDrive/ai_spring_2026/assignment2_interval_outputs/best_clips_by_query.parquet
Rows: 65


,query_image,youtube_url
0,query_0:left_mirror,https://www.youtube.com/embed/YcvECxtXoxQ?start=1163&end=1208
1,query_1:left_mirror,https://www.youtube.com/embed/YcvECxtXoxQ?start=1163&end=1208
2,query_2:wheel,https://www.youtube.com/embed/YcvECxtXoxQ?start=1323&end=1373
3,query_3:left_mirror,https://www.youtube.com/embed/YcvECxtXoxQ?start=1163&end=1208
4,query_4:left_mirror,https://www.youtube.com/embed/YcvECxtXoxQ?start=1163&end=1208
...,...,...
60,query_60:left_mirror,https://www.youtube.com/embed/YcvECxtXoxQ?start=1163&end=1208
61,query_61:wheel,https://www.youtube.com/embed/YcvECxtXoxQ?start=1323&end=1373
62,query_62:wheel,https://www.youtube.com/embed/YcvECxtXoxQ?start=1323&end=1373
63,query_63:wheel,https://www.youtube.com/embed/YcvECxtXoxQ?start=1323&end=1373


In [21]:

import os

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
else:
    print("Drive already mounted.")

OUT_DIR = globals().get("OUT_DIR", "/content/drive/MyDrive/ai_spring_2026/assignment2_interval_outputs")
os.makedirs(OUT_DIR, exist_ok=True)
print("OUT_DIR:", OUT_DIR)

DETECTIONS_PARQUET = globals().get("DETECTIONS_PARQUET", os.path.join(OUT_DIR, "detections_by_frame.parquet"))
CLIPS_PARQUET = globals().get("CLIPS_PARQUET", os.path.join(OUT_DIR, "clips_by_query.parquet"))
BEST_CLIPS_PARQUET = globals().get("BEST_CLIPS_PARQUET", os.path.join(OUT_DIR, "best_clips_by_query.parquet"))

print("DETECTIONS_PARQUET:", DETECTIONS_PARQUET)
print("CLIPS_PARQUET:", CLIPS_PARQUET)
print("BEST_CLIPS_PARQUET:", BEST_CLIPS_PARQUET)

import pandas as pd

det_df_candidates = ["det", "detections_by_frame", "df_frames", "detections_df"]
clips_df_candidates = ["clips", "clips_df", "clips_by_query"]
best_df_candidates = ["best_clips", "best_clips_df", "clips_best"]

def _first_existing_df(names):
    for n in names:
        v = globals().get(n, None)
        if isinstance(v, pd.DataFrame):
            return n, v
    return None, None

name_det, df_det = _first_existing_df(det_df_candidates)
name_clips, df_clips = _first_existing_df(clips_df_candidates)
name_best, df_best = _first_existing_df(best_df_candidates)

if df_det is not None:
    df_det.to_parquet(DETECTIONS_PARQUET, index=False)
    print(f"Saved {name_det} ->", DETECTIONS_PARQUET)
else:
    print("No in-memory detections DataFrame found (will rely on existing Parquet if present).")

if df_clips is not None:
    df_clips.to_parquet(CLIPS_PARQUET, index=False)
    print(f"Saved {name_clips} ->", CLIPS_PARQUET)
else:
    print("No in-memory clips DataFrame found (will rely on existing Parquet if present).")

if df_best is not None:
    df_best.to_parquet(BEST_CLIPS_PARQUET, index=False)
    print(f"Saved {name_best} ->", BEST_CLIPS_PARQUET)
else:
    print("No in-memory best-clips DataFrame found (will rely on existing Parquet if present).")

for p in [DETECTIONS_PARQUET, CLIPS_PARQUET, BEST_CLIPS_PARQUET]:
    print(os.path.exists(p), p)


Drive already mounted.
OUT_DIR: /content/drive/MyDrive/ai_spring_2026/assignment2_interval_outputs
DETECTIONS_PARQUET: /content/drive/MyDrive/ai_spring_2026/assignment2_interval_outputs/detections_by_frame.parquet
CLIPS_PARQUET: /content/drive/MyDrive/ai_spring_2026/assignment2_interval_outputs/clips_by_query.parquet
BEST_CLIPS_PARQUET: /content/drive/MyDrive/ai_spring_2026/assignment2_interval_outputs/best_clips_by_query.parquet
Saved det -> /content/drive/MyDrive/ai_spring_2026/assignment2_interval_outputs/detections_by_frame.parquet
Saved clips -> /content/drive/MyDrive/ai_spring_2026/assignment2_interval_outputs/clips_by_query.parquet
Saved best_clips -> /content/drive/MyDrive/ai_spring_2026/assignment2_interval_outputs/best_clips_by_query.parquet
True /content/drive/MyDrive/ai_spring_2026/assignment2_interval_outputs/detections_by_frame.parquet
True /content/drive/MyDrive/ai_spring_2026/assignment2_interval_outputs/clips_by_query.parquet
True /content/drive/MyDrive/ai_spring_2026/

In [23]:
import pandas as pd
from IPython.display import display, HTML

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 0)

def show_parquet(path, title):
    print("\n" + "="*80)
    print(title)
    print("Path:", path)
    df = pd.read_parquet(path)
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())
    display(df.head(len(df)))
    return df

best_df = show_parquet(BEST_CLIPS_PARQUET, "best_clips_by_query.parquet (one URL per query)")



best_clips_by_query.parquet (one URL per query)
Path: /content/drive/MyDrive/ai_spring_2026/assignment2_interval_outputs/best_clips_by_query.parquet
Rows: 65
Columns: ['query_image', 'youtube_url']


,query_image,youtube_url
0,query_0:left_mirror,https://www.youtube.com/embed/YcvECxtXoxQ?start=1163&end=1208
1,query_1:left_mirror,https://www.youtube.com/embed/YcvECxtXoxQ?start=1163&end=1208
2,query_2:wheel,https://www.youtube.com/embed/YcvECxtXoxQ?start=1323&end=1373
3,query_3:left_mirror,https://www.youtube.com/embed/YcvECxtXoxQ?start=1163&end=1208
4,query_4:left_mirror,https://www.youtube.com/embed/YcvECxtXoxQ?start=1163&end=1208
...,...,...
60,query_60:left_mirror,https://www.youtube.com/embed/YcvECxtXoxQ?start=1163&end=1208
61,query_61:wheel,https://www.youtube.com/embed/YcvECxtXoxQ?start=1323&end=1373
62,query_62:wheel,https://www.youtube.com/embed/YcvECxtXoxQ?start=1323&end=1373
63,query_63:wheel,https://www.youtube.com/embed/YcvECxtXoxQ?start=1323&end=1373


In [24]:
from huggingface_hub import notebook_login, create_repo, upload_file, whoami
import os

notebook_login()

username = whoami()["name"]
repo_id = globals().get("repo_id", f"{username}/rav4-exterior-retrieval")
create_repo(repo_id, repo_type="dataset", exist_ok=True)

upload_file(
    path_or_fileobj=DETECTIONS_PARQUET,
    path_in_repo="detections_by_frame.parquet",
    repo_id=repo_id,
    repo_type="dataset",
)

upload_file(
    path_or_fileobj=CLIPS_PARQUET,
    path_in_repo="clips_by_query.parquet",
    repo_id=repo_id,
    repo_type="dataset",
)

upload_file(
    path_or_fileobj=BEST_CLIPS_PARQUET,
    path_in_repo="best_clips_by_query.parquet",
    repo_id=repo_id,
    repo_type="dataset",
)

print("Uploaded repo:", f"https://huggingface.co/datasets/{repo_id}")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...tections_by_frame.parquet: 100%|##########| 19.8kB / 19.8kB            

No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...st_clips_by_query.parquet: 100%|##########| 2.47kB / 2.47kB            

Uploaded repo: https://huggingface.co/datasets/Naman02/rav4-exterior-retrieval-interval2
